In [1]:
import sys
import os
from pathlib import Path

notebook_dir = Path(os.getcwd())
project_root = notebook_dir.parent
src_dir      = project_root / "src"

if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from config_loader import load_config
import triage_filter

cfg = load_config()

# Resolve triage path (new key — created by run_triage if absent)
triage_path = Path(cfg["paths"]["triage"])

print("Config loaded.")
print("  raw_images :", cfg["paths"]["raw_images"])
print("  triage     :", triage_path)


Config loaded.
  raw_images : /mnt/storage_11tb/Drive_files_to_syncronize/3 - Images DataSets & Labelling Outputs/1639_DS/raw
  triage     : /mnt/storage_11tb/Drive_files_to_syncronize/3 - Images DataSets & Labelling Outputs/1639_DS/triage


# Stage 00a.2 — SigLIP Triage Filter

**What this notebook does:**  
Scores every raw patent image with [SigLIP](https://huggingface.co/timm/ViT-SO400M-14-SigLIP-384) (a large vision-language model) to detect images that are **not technical drawings** — tables, text pages, data grids, title sheets — before those images enter the expensive labelling pipeline downstream.

It is **completely non-destructive**: nothing in `raw/` is touched. Outputs are JSON/CSV manifests only.

---

## Why this step exists

PatSeer downloads *everything* on the Drawings tab — which often includes:
- Cover pages / title sheets
- Tables of component labels
- Text-only description pages
- Flowcharts and block diagrams that look like drawings but aren't figure crops

Running `00b` (figure crop + label matching) on those wastes GPU time and pollutes the labelled dataset. This step flags them so Stage `00b` can skip them.

---

## How the scoring works

Each image is scored against two zero-shot text prompts using SigLIP:

| Prompt | Meaning |
|--------|---------|
| `"a technical patent line drawing of an aircraft or mechanical component"` | → `score_drawing` |
| `"a table, chart, text page, or data grid with no technical drawing"` | → `score_table` |

The two scores are softmax-normalised so they sum to 1.  
`keep = True` when `score_drawing ≥ threshold` (default **0.60**).  
`keep = False` = flagged as non-drawing → will be skipped by `00b`.

> **Threshold guide:**  
> `0.60` (default) — conservative, keeps borderline images, low false-positive rate  
> `0.65` — moderate filtering  
> `0.70` — aggressive, only keep high-confidence drawings

---

## Outputs

Both written under `cfg["paths"]["triage"]` = `1639_DS/triage/`:

| File | Content |
|------|---------|
| `<patent_id>.json` | Per-image scores for one patent: `file`, `pred`, `score_drawing`, `score_table`, `keep` |
| `triage_summary.csv` | One row per patent: `patent_id`, `total_images`, `flagged_count`, `flagged_ratio` |

Already-processed patents are **not re-scored** on repeat runs (JSON exists → skip).

---

## Where it fits in the pipeline

```
00a    (download from PatSeer)
 ↓
00a.1  (audit: which patents are missing / need re-download)
 ↓
00a.2  ← YOU ARE HERE  (score every image: drawing vs. non-drawing)
 ↓
00b    (figure crop + _F/_Fu label matching — uses triage to skip flagged images)
 ↓
02     (pad + resize to 518×518)
```

---

## Cell guide

| Cell | What it does |
|------|--------------|
| 1 | Load config — resolves `raw_images` and `triage` paths |
| 2 | Load SigLIP model (~3 GB weights, cached after first run, needs GPU) |
| 3 | Run triage — set `limit=10` to test on 10 patents first, `limit=None` for full dataset |
| 4 | Inspect results — flagged ratio summary + per-image table for worst patents |

---

## Before you run

- **GPU required** — SigLIP ViT-SO400M is large; CPU inference is very slow (~10 s/image).  
- **First run downloads ~3 GB** of weights into the HuggingFace cache (`~/.cache/huggingface/`).  
- Run Cell 3 with `limit=10` first to check the threshold is sensible for your data before processing all 1350 folders.  
- If scores cluster around 0.50–0.52 (as in the test output above), the prompts may need tuning for your specific patent images — see `src/triage_filter.py` → `_PROMPT_KEEP` / `_PROMPT_FLAG`.

In [2]:
# Loads SigLIP ViT-SO400M-14-SigLIP-384 via open_clip.
# Downloads ~3 GB of weights on first run (cached by HuggingFace Hub).
model, processor, device = triage_filter.load_siglip_model(cfg)
print(f"SigLIP ready on: {device}")


[SigLIP] Loaded ViT-SO400M-14-SigLIP-384 on cuda
SigLIP ready on: cuda


In [ ]:
# ── Run options ──────────────────────────────────────────────────────────────
TEST_MODE  = True   # True = only 10 patents (quick sanity check)
                    # False = full dataset (all ~1350 folders)
THRESHOLD  = 0.50   # kept for API compatibility; logic now uses pure argmax
                    # (discard only when model prefers the table/text prompt)
# ─────────────────────────────────────────────────────────────────────────────

limit = 600 if TEST_MODE else None
print(f"Running triage: {'TEST MODE (10 patents)' if TEST_MODE else 'FULL DATASET'}, threshold={THRESHOLD}")
triage_filter.run_triage(cfg, threshold=THRESHOLD, limit=limit)

Running triage: TEST MODE (10 patents), threshold=0.5
[SigLIP] Loaded ViT-SO400M-14-SigLIP-384 on cuda
  ✓ AU2021232803A1_85775850  total=10  flagged=0
  ✓ AU2023270571A1_95939416  total=4  flagged=0
  ✓ BR112023017877B1_83067308  total=7  flagged=0
  ✓ CA2963502A1_63709012  total=2  flagged=0
  ✓ CA2967402A1_58709316  total=5  flagged=0
  ✓ CN103786881A_50663046  total=12  flagged=0
  ✓ CN104229137A_52218225  total=9  flagged=3
  ✓ CN105730692A_56254804  total=5  flagged=0
  ✓ CN105799934A_56467513  total=9  flagged=3
  ✓ CN105836109A_56597035  total=4  flagged=4
  ✓ CN105923154A_56832133  total=2  flagged=0
  ✓ CN106005373A_57096574  total=12  flagged=0
  ✓ CN106005387A_57116011  total=6  flagged=2
  ✓ CN106043685A_57172473  total=3  flagged=0
  ✓ CN106218887A_57553770  total=3  flagged=0
  ✓ CN106314794A_57819675  total=7  flagged=5
  ✓ CN106628168A_58813122  total=6  flagged=1
  ✓ CN106864746A_59166843  total=3  flagged=0
  ✓ CN106882372A_59180265  total=5  flagged=0
  ✓ CN10692704

/home/vasco/anaconda3/envs/doclayout_yolo2/lib/python3.10/site-packages/PIL/Image.py:1137: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


  ✓ EP3290334A1_59745849  total=12  flagged=2
  ✓ EP3360780A1_61132051  total=12  flagged=0
  ✓ EP3453616A1_63293968  total=2  flagged=0
  ✓ EP3594113A1_63449420  total=10  flagged=1
  ✓ EP3636545A1_68242353  total=3  flagged=0
  ✓ EP3770063A1_67851083  total=8  flagged=1
  ✓ EP3805100A1_68242276  total=8  flagged=0
  ✓ EP3907132A1_70680379  total=8  flagged=3
  ✓ EP3974315A1_72659736  total=6  flagged=0
  ✓ EP3998191A1_74672189  total=7  flagged=0
  ✓ EP3998195A1_74672222  total=5  flagged=0
  ✓ EP3998199A1_74672090  total=6  flagged=0
  ✓ EP3998206A1_74672197  total=8  flagged=1
  ✓ EP3998209A1_74859722  total=12  flagged=1
  ✓ EP4134301A1_77316847  total=6  flagged=0
  ✓ EP4151521A1_78695660  total=8  flagged=1
  ✓ EP4155193A1_77913035  total=7  flagged=1
  ✓ EP4230518A1_80445889  total=6  flagged=1
  ✓ EP4303124A1_82850459  total=4  flagged=0
  ✓ EP4306409A1_82594614  total=3  flagged=0
  ✓ EP4339109A1_83593899  total=8  flagged=0
  ✓ EP4361026A1_84043779  total=7  flagged=1
  ✓ EP

## Cell 3b — Re-threshold existing results (no GPU needed)

Already-confirmed discards (`keep=False`) are **never** upgraded back.  
Only images currently `keep=True` are re-evaluated against the new threshold.  
Rewrites all JSONs and `triage_summary.csv` in-place.

In [ ]:
import importlib, triage_filter
importlib.reload(triage_filter)   # pick up src edits without restarting kernel

# ── Choose mode ───────────────────────────────────────────────────────────────
#
#   "add"   — rethreshold_existing: only ADDS new discards, never re-enables
#             anything already marked keep=False (safe, one-way)
#
#   "reset" — reset_threshold: re-derives keep from raw scores for ALL images,
#             works in both directions — use this to undo an over-aggressive run
#
MODE      = "reset"   # "add" | "reset"
THRESHOLD = 0.50      # tune this value
# ─────────────────────────────────────────────────────────────────────────────

if MODE == "add":
    triage_filter.rethreshold_existing(cfg, new_threshold=THRESHOLD)
elif MODE == "reset":
    triage_filter.reset_threshold(cfg, threshold=THRESHOLD)
else:
    print(f"Unknown MODE={MODE!r} — choose 'add' or 'reset'")


In [ ]:
import json
import pandas as pd

triage_path = Path(cfg["paths"]["triage"])
summary_csv = triage_path / "triage_summary.csv"

if not summary_csv.exists():
    print("No triage_summary.csv found — run Cell 3 first.")
else:
    summary_df = pd.read_csv(summary_csv)

    total_images  = summary_df["total_images"].sum()
    total_flagged = summary_df["flagged_count"].sum()
    overall_ratio = total_flagged / max(1, total_images)

    print(f"Total images scored : {total_images}")
    print(f"Total flagged       : {total_flagged}")
    print(f"Overall flagged ratio: {overall_ratio:.1%}")
    print()

    # Show per-figure results for the 3 patents with the most flagged images
    top3 = summary_df.nlargest(3, "flagged_count")[["patent_id", "total_images", "flagged_count", "flagged_ratio"]]
    print("Top 3 patents by flagged count:")
    display(top3.reset_index(drop=True))
    print()

    for patent_id in top3["patent_id"]:
        json_path = triage_path / f"{patent_id}.json"
        if not json_path.exists():
            print(f"  {patent_id}: JSON not found")
            continue
        with open(json_path) as fh:
            data = json.load(fh)
        figures_df = pd.DataFrame(data["figures"])
        print(f"\n--- {patent_id}  (total={data['total']}  flagged={data['flagged']}) ---")
        display(figures_df[["file", "pred", "score_drawing", "score_table", "keep"]])


## Cell 5 — Visual inspection: kept vs. discarded images

Shows the actual images for one patent so you can judge whether the threshold is working correctly.  
- **Green border** = kept (`keep=True`, `score_drawing ≥ threshold`)  
- **Red border** = discarded (`keep=False`)  

Change `INSPECT_PATENT` to any patent ID from the triage results, or leave it as `None` to auto-pick the first processed patent.

In [ ]:
import json
import math
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import ipywidgets as widgets
from IPython.display import display, clear_output
from pathlib import Path
from PIL import Image

triage_path = Path(cfg["paths"]["triage"])
raw_dir     = Path(cfg["paths"]["raw_images"])
MAX_COLS    = 4

# All processed patents, sorted
patent_ids = sorted(p.stem for p in triage_path.glob("*.json")
                    if p.stem != "triage_summary")
if not patent_ids:
    print("No triage JSON files found — run Cell 3 first.")
    raise SystemExit

print(f"Found {len(patent_ids)} processed patents. Use the buttons or dropdown to navigate.")

# ── Widgets ───────────────────────────────────────────────────────────────────
idx_state   = {"i": 0, "updating": False}   # flag prevents dropdown re-entrancy
out         = widgets.Output()
btn_prev    = widgets.Button(description="◀ Prev", button_style="info",  layout=widgets.Layout(width="120px"))
btn_next    = widgets.Button(description="Next ▶", button_style="info",  layout=widgets.Layout(width="120px"))
lbl_pos     = widgets.Label(value=f"1 / {len(patent_ids)}")
patent_dd   = widgets.Dropdown(options=patent_ids, value=patent_ids[0],
                                layout=widgets.Layout(width="360px"))

def show_patent(patent_id):
    json_path = triage_path / f"{patent_id}.json"
    with open(json_path) as fh:
        data = json.load(fh)
    figures = data["figures"]
    n       = len(figures)
    if n == 0:
        print(f"{patent_id}: no figures in triage JSON")
        return
    n_kept  = sum(1 for f in figures if f["keep"])
    n_disc  = n - n_kept

    n_cols = min(MAX_COLS, n)
    n_rows = math.ceil(n / n_cols)
    fig, axes = plt.subplots(n_rows, n_cols,
                             figsize=(n_cols * 3.5, n_rows * 3.8),
                             squeeze=False)
    flat_axes = [ax for row in axes for ax in row]

    patent_dir = raw_dir / patent_id
    for ax, fig_info in zip(flat_axes, figures):
        img_path = patent_dir / fig_info["file"]
        try:
            img = Image.open(img_path).convert("RGB")
            ax.imshow(img)
        except Exception:
            ax.text(0.5, 0.5, "not found", ha="center", va="center",
                    color="grey", transform=ax.transAxes)
        keep  = fig_info["keep"]
        color = "#2ecc71" if keep else "#e74c3c"
        label = "KEEP" if keep else "DISCARD"
        for spine in ax.spines.values():
            spine.set_edgecolor(color)
            spine.set_linewidth(4)
        ax.set_title(f"{fig_info['file']}\n{label}  {fig_info['score_drawing']:.3f}",
                     fontsize=7, color=color, pad=3)
        ax.set_xticks([]); ax.set_yticks([])

    for ax in flat_axes[n:]:
        ax.axis("off")

    keep_patch    = mpatches.Patch(color="#2ecc71", label=f"Kept ({n_kept})")
    discard_patch = mpatches.Patch(color="#e74c3c", label=f"Discarded ({n_disc})")
    fig.legend(handles=[keep_patch, discard_patch], loc="lower center",
               ncol=2, fontsize=10, frameon=False, bbox_to_anchor=(0.5, -0.01))
    fig.suptitle(f"{patent_id}  —  {n_kept}/{n} kept  (threshold={THRESHOLD})",
                 fontsize=12, fontweight="bold", y=1.01)
    plt.tight_layout()
    plt.show()

def refresh():
    i = idx_state["i"]
    # Update label and dropdown without triggering the observer callback
    lbl_pos.value = f"{i+1} / {len(patent_ids)}"
    idx_state["updating"] = True
    patent_dd.value = patent_ids[i]
    idx_state["updating"] = False
    with out:
        clear_output(wait=True)
        show_patent(patent_ids[i])

def on_prev(_):
    if idx_state["i"] > 0:
        idx_state["i"] -= 1
        refresh()

def on_next(_):
    if idx_state["i"] < len(patent_ids) - 1:
        idx_state["i"] += 1
        refresh()

def on_dropdown(change):
    # Only react to user-driven value changes, not programmatic ones from refresh()
    if change["name"] == "value" and not idx_state["updating"]:
        new_val = change["new"]
        if new_val in patent_ids:
            idx_state["i"] = patent_ids.index(new_val)
            refresh()

btn_prev.on_click(on_prev)
btn_next.on_click(on_next)
patent_dd.observe(on_dropdown)

nav_bar = widgets.HBox([btn_prev, lbl_pos, btn_next, patent_dd])
display(nav_bar, out)
refresh()

## Cell 6 — All discarded images (across all patents)

Browses every image flagged as **discarded** (`keep=False`) across the full triage run, paginated in groups of `PAGE_SIZE`.  
Useful for quickly auditing false positives — images the model rejected that look like valid drawings.

In [ ]:
import json
import math
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
from pathlib import Path
from PIL import Image

triage_path = Path(cfg["paths"]["triage"])
raw_dir     = Path(cfg["paths"]["raw_images"])

PAGE_SIZE = 32   # images per page
MAX_COLS  = 4

# Collect all discarded images across every processed patent
discarded = []   # list of (patent_id, file, score_drawing, score_table)
for json_path in sorted(triage_path.glob("*.json")):
    if json_path.stem == "triage_summary":
        continue
    with open(json_path) as fh:
        data = json.load(fh)
    patent_id = json_path.stem
    for fig in data["figures"]:
        if not fig["keep"]:
            discarded.append((patent_id, fig["file"],
                               fig["score_drawing"], fig["score_table"]))

n_total = len(discarded)
n_pages = max(1, math.ceil(n_total / PAGE_SIZE))
print(f"Total discarded images: {n_total}  ({n_pages} pages of {PAGE_SIZE})")

# ── Widgets ───────────────────────────────────────────────────────────────────
page_state = {"p": 0, "updating": False}
out_disc   = widgets.Output()
btn_dp     = widgets.Button(description="◀ Prev", button_style="warning", layout=widgets.Layout(width="120px"))
btn_dn     = widgets.Button(description="Next ▶", button_style="warning", layout=widgets.Layout(width="120px"))
lbl_dp     = widgets.Label(value=f"Page 1 / {n_pages}")

def show_page(p):
    batch = discarded[p * PAGE_SIZE : (p + 1) * PAGE_SIZE]
    n     = len(batch)
    n_cols = min(MAX_COLS, n)
    n_rows = math.ceil(n / max(n_cols, 1))

    fig, axes = plt.subplots(n_rows, n_cols,
                             figsize=(n_cols * 3.5, n_rows * 3.8),
                             squeeze=False)
    flat_axes = [ax for row in axes for ax in row]

    for ax, (patent_id, fname, s_draw, s_table) in zip(flat_axes, batch):
        img_path = raw_dir / patent_id / fname
        try:
            img = Image.open(img_path).convert("RGB")
            ax.imshow(img)
        except Exception:
            ax.text(0.5, 0.5, "not found", ha="center", va="center",
                    color="grey", transform=ax.transAxes)
        for spine in ax.spines.values():
            spine.set_edgecolor("#e74c3c")
            spine.set_linewidth(3)
        ax.set_title(f"{patent_id}\n{fname}\ndraw={s_draw:.3f}  tbl={s_table:.3f}",
                     fontsize=6, color="#e74c3c", pad=3)
        ax.set_xticks([]); ax.set_yticks([])

    for ax in flat_axes[n:]:
        ax.axis("off")

    fig.suptitle(f"Discarded images — page {p+1}/{n_pages}  "
                 f"(images {p*PAGE_SIZE+1}–{p*PAGE_SIZE+n} of {n_total})",
                 fontsize=11, fontweight="bold")
    plt.tight_layout()
    plt.show()

def refresh_disc(new_p):
    if page_state["updating"]:
        return
    page_state["updating"] = True
    page_state["p"] = new_p
    lbl_dp.value = f"Page {new_p + 1} / {n_pages}"
    page_state["updating"] = False
    with out_disc:
        clear_output(wait=True)
        show_page(new_p)

def on_dp(_):
    p = page_state["p"]
    if p > 0:
        refresh_disc(p - 1)

def on_dn(_):
    p = page_state["p"]
    if p < n_pages - 1:
        refresh_disc(p + 1)

btn_dp.on_click(on_dp)
btn_dn.on_click(on_dn)

nav_disc = widgets.HBox([btn_dp, lbl_dp, btn_dn])
display(nav_disc, out_disc)
refresh_disc(0)
